In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lit, current_timestamp

def generate_streaming_data(source_catalog_name:str, target_catalog_name:str, bronze_schema:str, source_name:str, table_name:str, num_files:int=20, num_rows_per_file:int=1000):
    """
    Function which generates streaming data based on a source and table names

    Arguments:
    target_catalog_name (string): Name of the catalog, i.e. wanderbricks
    source_catalog_name (string): Name of the catalog, i.e. samples
    bronze_schema (string): Name of the bronze schema, i.e. bronze
    source_name (string): Name of the source, i.e. wanderbricks
    table_name (string): Name of the table, i.e. bookings
    num_files (int, optional): Number of files to generate
    num_rows_per_file (int, optional): Number of rows per file

    Returns:
    landing_path (string): Path to landing directory
    checkpoint_path (string): Path to checkpoint directory
    schema_path (string): Path to schema directory
    """

    # Create volume if doesn't exists yet
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{target_catalog_name}`.`{bronze_schema}`.`{source_name}`")

    landing_path = f"/Volumes/{target_catalog_name}/{bronze_schema}/{source_name}/landing/{table_name}"
    checkpoint_path = f"/Volumes/{target_catalog_name}/{bronze_schema}/{source_name}/checkpoint/{bronze_schema}_{table_name}"
    schema_path = f"/Volumes/{target_catalog_name}/{bronze_schema}/{source_name}/schemas/{bronze_schema}_{table_name}"

    df = spark.table(f"`{source_catalog_name}`.`{source_name}`.`{table_name}`")\
              .withColumn("_rn", row_number().over(Window.orderBy(lit(1))))

    for batch_id in range(num_files):
        start = batch_id * num_rows_per_file
        end = start + num_rows_per_file

        (
            df
            .filter((df["_rn"] > start) & (df["_rn"] <= end))
            .drop("_rn")
            .withColumn("_source_batch_id", lit(batch_id))
            .withColumn("_landing_created_at", current_timestamp())
            .write
            .mode("append")
            .json(f"{landing_path}/batch_id={batch_id}")
        )

    return landing_path, checkpoint_path, schema_path